In [45]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
df=pd.read_csv("data.csv",encoding="cp1252")
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-10 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-01-10 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-10 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-10 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-10 08:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12-09-11 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12-09-11 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12-09-11 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12-09-11 12:50,4.15,12680.0,France


In [47]:
df.isnull().sum()
df.dropna(inplace=True)
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [48]:
df.shape

(406829, 8)

In [ ]:
# Conversion purposes

df["CustomerID"]=df["CustomerID"].astype(int)         # it will change string invoice date to date time format
df["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"])     # max will give last date of purchasee --- so take last 1 day
todays_date=df["InvoiceDate"].max()+pd.Timedelta(days=1)     # then only last purcahse date also 1 day result gave

C:\Users\user\AppData\Local\Temp\ipykernel_9796\2773300622.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"])


In [ ]:
# for Recency ,Frequency , Moneyty

# -----------Invoicedate is for ____Recency

# -----------Quantity * Unitprice = Totalprice is for ____Monetry

# -----------InvoiceNumber is for ____Frequency



In [51]:
# Recency Filter

df=df[    (df["CustomerID"].notnull()       &      (df["Quantity"]>0))   ]
df["Totalprice"]=df["Quantity"]*df["UnitPrice"]
df

C:\Users\user\AppData\Local\Temp\ipykernel_9796\395659976.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Totalprice"]=df["Quantity"]*df["UnitPrice"]


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Totalprice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60


In [52]:
# R_F_M

rfm=df.groupby("CustomerID").agg({"InvoiceDate":lambda x : (todays_date - x.max()).days,"InvoiceNo":"nunique","Totalprice":"sum"})
rfm

,InvoiceDate,InvoiceNo,Totalprice
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1797.24
12349,19,1,1757.55
12350,310,1,334.40
...,...,...,...
18280,278,1,180.60
18281,181,1,80.82
18282,8,2,178.05


In [53]:
rfm.columns=["Recency","Frequency","Moneytry"]
rfm

,Recency,Frequency,Moneytry
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1797.24
12349,19,1,1757.55
12350,310,1,334.40
...,...,...,...
18280,278,1,180.60
18281,181,1,80.82
18282,8,2,178.05


In [56]:
# Spending Pattern

sp=df.groupby("CustomerID").agg({"Quantity":["sum","mean","max"],"Totalprice":["sum","mean","max"],})
sp

Quantity                      Totalprice                        
                sum          mean    max        sum          mean       max
CustomerID                                                                 
12346         74215  74215.000000  74215   77183.60  77183.600000  77183.60
12347          2458     13.505495    240    4310.00     23.681319    249.60
12348          2341     75.516129    144    1797.24     57.975484    240.00
12349           631      8.643836     36    1757.55     24.076027    300.00
12350           197     11.588235     24     334.40     19.670588     40.00
...             ...           ...    ...        ...           ...       ...
18280            45      4.500000      8     180.60     18.060000     23.70
18281            54      7.714286     12      80.82     11.545714     16.95
18282           103      8.583333     48     178.05     14.837500     25.50
18283          1397      1.847884     13    2094.88      2.771005     20.80
18287          1586     22.657143     60    1837.28     26.246857     87.00

[4339 rows x 6 columns]

In [57]:
sp.columns=["Totalqty","avgqty","maxqty","Totalspend","avgspend","maxspend"]
sp

,Totalqty,avgqty,maxqty,Totalspend,avgspend,maxspend
CustomerID,,,,,,
12346,74215,74215.000000,74215,77183.60,77183.600000,77183.60
12347,2458,13.505495,240,4310.00,23.681319,249.60
12348,2341,75.516129,144,1797.24,57.975484,240.00
12349,631,8.643836,36,1757.55,24.076027,300.00
12350,197,11.588235,24,334.40,19.670588,40.00
...,...,...,...,...,...,...
18280,45,4.500000,8,180.60,18.060000,23.70
18281,54,7.714286,12,80.82,11.545714,16.95
18282,103,8.583333,48,178.05,14.837500,25.50


In [58]:
pd.merge(rfm,sp,on="CustomerID",how="left")

,Recency,Frequency,Moneytry,Totalqty,avgqty,maxqty,Totalspend,avgspend,maxspend
CustomerID,,,,,,,,,
12346,326,1,77183.60,74215,74215.000000,74215,77183.60,77183.600000,77183.60
12347,2,7,4310.00,2458,13.505495,240,4310.00,23.681319,249.60
12348,75,4,1797.24,2341,75.516129,144,1797.24,57.975484,240.00
12349,19,1,1757.55,631,8.643836,36,1757.55,24.076027,300.00
12350,310,1,334.40,197,11.588235,24,334.40,19.670588,40.00
...,...,...,...,...,...,...,...,...,...
18280,278,1,180.60,45,4.500000,8,180.60,18.060000,23.70
18281,181,1,80.82,54,7.714286,12,80.82,11.545714,16.95
18282,8,2,178.05,103,8.583333,48,178.05,14.837500,25.50
